In [0]:
df_bronze = spark.read.table("hotel.bronze.hotel_bookings_raw")
display(df_bronze)

booking_id,hotel_id,hotel_city,customer_id,customer_name,customer_email,check_in_date,check_out_date,room_type,num_guests,total_amount,currency,booking_status
bc3c2ecb-89b3-4f87-9714-355c5088b35d,1860,Michelleport,1d541501-4a4a-479e-9d6c-91f80cf8452e,Tracy Ortega,invalid-email,1/11/2026,2026-01-19,Deluxe,4,252.49,USD,Confirmed
6834f794-1cee-41fe-ba24-d54368d6865e,1058,North Tina,11e3742c-0326-473e-964c-db5ab6de07f2,Karen Coleman,invalid-email,10/16/2025,2025-10-20,Suite,2,150.48,INR,Confirmed
e8f146a0-faf2-4c69-9a2b-ce6593d10950,1638,New Frederickville,7abf382a-89b8-4ed5-b5db-fe8c9b66f979,Anthony Alvarez,invalid-email,2/6/2026,2026-02-12,Standard,5,197.22,USD,No-Show
83bd8eab-d136-4153-bb30-4d0367b1e106,1980,Port Wesley,917c92b9-9ef1-46fc-9098-db84612c8856,Russell Williams,invalid-email,11/11/2025,2025-11-17,Deluxe,2,464.3,EUR,No-Show
c9a791c0-1f34-4c18-b1e6-f1c85ab916c4,1739,North Stevehaven,17485b29-a80e-41a4-8d3a-e783350006f7,Natalie Green,robert19@example.org,2/19/2025,2025-02-27,Standard,2,507.24,EUR,Confirmed
fadd2f9a-ee13-4e4a-98bf-a685669a40d0,1923,Cliffordbury,451192ab-3d5c-480a-b702-ded5aafcbdaa,Jean Chan,tonyaellison@example.com,5/28/2025,2025-06-02,Standard,2,503.82,INR,Confirmeeed
0e62ae16-3c2c-4eb9-ba55-89e6f8e4f3dc,1136,Martintown,707d69f1-a8d5-4665-a092-68fa6b5c64c8,Aaron Young,michael60@example.net,3/11/2025,2025-03-19,Deluxe,1,176.15,USD,Confirmed
3ac87823-3996-49b9-bde7-b9c30ab9cc44,1299,Johnsonland,0567969e-3d96-40f9-9f28-54a917f9c7c6,Yvette Butler,invalid-email,5/23/2026,2026-05-26,Suite,4,470.29,INR,Confirmed
c04fd7c4-6cba-4402-8c3f-fa9e82c54b3d,1348,West Jenniferborough,aad9ddf8-6514-41ed-bcae-d4de96982a40,David Hall,salastina@example.com,1/30/2026,2026-02-07,Deluxe,2,-506.82,USD,Cancelled
89c4d14c-a632-4d64-95ca-8a1f44f7ed96,1564,South Grant,983b9421-baa4-4ad1-8a63-11b5d03b6a10,Sarah Johnson,swoods@example.com,5/21/2026,2026-05-25,Suite,3,355.99,INR,Confirmed


In [0]:

# Null Value
from pyspark.sql.functions import col, sum

null_counts = df_bronze.agg(*[sum(col(c).isNull().cast("int")).alias(c) for c in df_bronze.columns])
display(null_counts)

booking_id,hotel_id,hotel_city,customer_id,customer_name,customer_email,check_in_date,check_out_date,room_type,num_guests,total_amount,currency,booking_status
2,2,3,1,4,1,2,4,2,3,1,1,3


In [0]:
# Duplicate Record
from pyspark.sql.functions import count

duplicate_counts = df_bronze.groupBy(df_bronze.columns).agg(count("*").alias("count")).filter(col("count") > 1)
display(duplicate_counts)

booking_id,hotel_id,hotel_city,customer_id,customer_name,customer_email,check_in_date,check_out_date,room_type,num_guests,total_amount,currency,booking_status,count


In [0]:
df_bronze.printSchema()

root
 |-- booking_id: string (nullable = true)
 |-- hotel_id: integer (nullable = true)
 |-- hotel_city: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- customer_email: string (nullable = true)
 |-- check_in_date: string (nullable = true)
 |-- check_out_date: date (nullable = true)
 |-- room_type: string (nullable = true)
 |-- num_guests: integer (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- booking_status: string (nullable = true)



date format
10/16/2025
(MM/DD/YYYY)
(M/DD/YYYY)
__


In [0]:

df_bronze.select("check_in_date").dtypes

[('check_in_date', 'string')]

date format
2025-06-24
(YYYY-MM-DD)
__



In [0]:
df_bronze.select("check_out_date").dtypes

[('check_out_date', 'date')]

In [0]:
from pyspark.sql.functions import coalesce, try_to_date, col

df = df_bronze.withColumn(
    "check_in_date",
    coalesce(
        try_to_date(col("check_in_date"), "M/d/yyyy"),
        try_to_date(col("check_in_date"), "MM/d/yyyy"),


        try_to_date(col("check_in_date"), "dd-MM-yyyy"),
        try_to_date(col("check_in_date"), "MM-dd-yyyy"),
        try_to_date(col("check_in_date"), "yyyy-MM-dd"),
    )
)

display(df)

booking_id,hotel_id,hotel_city,customer_id,customer_name,customer_email,check_in_date,check_out_date,room_type,num_guests,total_amount,currency,booking_status
bc3c2ecb-89b3-4f87-9714-355c5088b35d,1860,Michelleport,1d541501-4a4a-479e-9d6c-91f80cf8452e,Tracy Ortega,invalid-email,2026-01-11,2026-01-19,Deluxe,4,252.49,USD,Confirmed
6834f794-1cee-41fe-ba24-d54368d6865e,1058,North Tina,11e3742c-0326-473e-964c-db5ab6de07f2,Karen Coleman,invalid-email,2025-10-16,2025-10-20,Suite,2,150.48,INR,Confirmed
e8f146a0-faf2-4c69-9a2b-ce6593d10950,1638,New Frederickville,7abf382a-89b8-4ed5-b5db-fe8c9b66f979,Anthony Alvarez,invalid-email,2026-02-06,2026-02-12,Standard,5,197.22,USD,No-Show
83bd8eab-d136-4153-bb30-4d0367b1e106,1980,Port Wesley,917c92b9-9ef1-46fc-9098-db84612c8856,Russell Williams,invalid-email,2025-11-11,2025-11-17,Deluxe,2,464.3,EUR,No-Show
c9a791c0-1f34-4c18-b1e6-f1c85ab916c4,1739,North Stevehaven,17485b29-a80e-41a4-8d3a-e783350006f7,Natalie Green,robert19@example.org,2025-02-19,2025-02-27,Standard,2,507.24,EUR,Confirmed
fadd2f9a-ee13-4e4a-98bf-a685669a40d0,1923,Cliffordbury,451192ab-3d5c-480a-b702-ded5aafcbdaa,Jean Chan,tonyaellison@example.com,2025-05-28,2025-06-02,Standard,2,503.82,INR,Confirmeeed
0e62ae16-3c2c-4eb9-ba55-89e6f8e4f3dc,1136,Martintown,707d69f1-a8d5-4665-a092-68fa6b5c64c8,Aaron Young,michael60@example.net,2025-03-11,2025-03-19,Deluxe,1,176.15,USD,Confirmed
3ac87823-3996-49b9-bde7-b9c30ab9cc44,1299,Johnsonland,0567969e-3d96-40f9-9f28-54a917f9c7c6,Yvette Butler,invalid-email,2026-05-23,2026-05-26,Suite,4,470.29,INR,Confirmed
c04fd7c4-6cba-4402-8c3f-fa9e82c54b3d,1348,West Jenniferborough,aad9ddf8-6514-41ed-bcae-d4de96982a40,David Hall,salastina@example.com,2026-01-30,2026-02-07,Deluxe,2,-506.82,USD,Cancelled
89c4d14c-a632-4d64-95ca-8a1f44f7ed96,1564,South Grant,983b9421-baa4-4ad1-8a63-11b5d03b6a10,Sarah Johnson,swoods@example.com,2026-05-21,2026-05-25,Suite,3,355.99,INR,Confirmed


In [0]:

# Null Value
from pyspark.sql.functions import col, sum

null_counts = df.agg(*[sum(col(c).isNull().cast("int")).alias(c) for c in df.columns])
display(null_counts)

booking_id,hotel_id,hotel_city,customer_id,customer_name,customer_email,check_in_date,check_out_date,room_type,num_guests,total_amount,currency,booking_status
2,2,3,1,4,1,196,4,2,3,1,1,3


In [0]:
# df.filter(col("check_in_date") > col("check_out_date")).show()

from pyspark.sql.functions import to_date

date_issue_df = df.filter(
    to_date(col("check_out_date")) < to_date(col("check_in_date"))
)

date_issue_df.select("check_in_date", "check_out_date").show()

+-------------+--------------+
|check_in_date|check_out_date|
+-------------+--------------+
|   2026-01-31|    2025-11-20|
|   2026-04-11|    2025-11-20|
|   2026-02-01|    2025-11-20|
|   2025-12-20|    2025-11-20|
|   2026-02-15|    2025-11-20|
|   2026-02-27|    2025-11-20|
|   2026-02-02|    2025-11-20|
|   2026-04-09|    2025-11-20|
|   2025-12-06|    2025-11-20|
|   2026-01-31|    2025-11-20|
|   2026-02-21|    2025-11-20|
|   2026-04-04|    2025-11-20|
|   2026-03-02|    2025-11-20|
|   2025-11-27|    2025-11-20|
|   2025-12-12|    2025-11-20|
|   2026-01-20|    2025-11-20|
|   2025-12-02|    2025-11-20|
|   2025-12-06|    2025-11-20|
|   2026-02-16|    2025-11-20|
|   2026-04-27|    2025-11-20|
+-------------+--------------+
only showing top 20 rows


In [0]:
from pyspark.sql.functions import col, when

# condition: check_in_date > check_out_date → set NULL
df = df.withColumn(
    "check_in_date",
    when(col("check_in_date") > col("check_out_date"), None)  # NULL assign
    .otherwise(col("check_in_date"))  # baki same value
)

display(df)

booking_id,hotel_id,hotel_city,customer_id,customer_name,customer_email,check_in_date,check_out_date,room_type,num_guests,total_amount,currency,booking_status
bc3c2ecb-89b3-4f87-9714-355c5088b35d,1860,Michelleport,1d541501-4a4a-479e-9d6c-91f80cf8452e,Tracy Ortega,invalid-email,2026-01-11,2026-01-19,Deluxe,4,252.49,USD,Confirmed
6834f794-1cee-41fe-ba24-d54368d6865e,1058,North Tina,11e3742c-0326-473e-964c-db5ab6de07f2,Karen Coleman,invalid-email,2025-10-16,2025-10-20,Suite,2,150.48,INR,Confirmed
e8f146a0-faf2-4c69-9a2b-ce6593d10950,1638,New Frederickville,7abf382a-89b8-4ed5-b5db-fe8c9b66f979,Anthony Alvarez,invalid-email,2026-02-06,2026-02-12,Standard,5,197.22,USD,No-Show
83bd8eab-d136-4153-bb30-4d0367b1e106,1980,Port Wesley,917c92b9-9ef1-46fc-9098-db84612c8856,Russell Williams,invalid-email,2025-11-11,2025-11-17,Deluxe,2,464.3,EUR,No-Show
c9a791c0-1f34-4c18-b1e6-f1c85ab916c4,1739,North Stevehaven,17485b29-a80e-41a4-8d3a-e783350006f7,Natalie Green,robert19@example.org,2025-02-19,2025-02-27,Standard,2,507.24,EUR,Confirmed
fadd2f9a-ee13-4e4a-98bf-a685669a40d0,1923,Cliffordbury,451192ab-3d5c-480a-b702-ded5aafcbdaa,Jean Chan,tonyaellison@example.com,2025-05-28,2025-06-02,Standard,2,503.82,INR,Confirmeeed
0e62ae16-3c2c-4eb9-ba55-89e6f8e4f3dc,1136,Martintown,707d69f1-a8d5-4665-a092-68fa6b5c64c8,Aaron Young,michael60@example.net,2025-03-11,2025-03-19,Deluxe,1,176.15,USD,Confirmed
3ac87823-3996-49b9-bde7-b9c30ab9cc44,1299,Johnsonland,0567969e-3d96-40f9-9f28-54a917f9c7c6,Yvette Butler,invalid-email,2026-05-23,2026-05-26,Suite,4,470.29,INR,Confirmed
c04fd7c4-6cba-4402-8c3f-fa9e82c54b3d,1348,West Jenniferborough,aad9ddf8-6514-41ed-bcae-d4de96982a40,David Hall,salastina@example.com,2026-01-30,2026-02-07,Deluxe,2,-506.82,USD,Cancelled
89c4d14c-a632-4d64-95ca-8a1f44f7ed96,1564,South Grant,983b9421-baa4-4ad1-8a63-11b5d03b6a10,Sarah Johnson,swoods@example.com,2026-05-21,2026-05-25,Suite,3,355.99,INR,Confirmed


In [0]:

# Null Value
from pyspark.sql.functions import col, sum

null_counts = df.agg(*[sum(col(c).isNull().cast("int")).alias(c) for c in df.columns])
display(null_counts)

booking_id,hotel_id,hotel_city,customer_id,customer_name,customer_email,check_in_date,check_out_date,room_type,num_guests,total_amount,currency,booking_status
2,2,3,1,4,1,267,4,2,3,1,1,3


In [0]:
# df.filter(col("check_in_date") > col("check_out_date")).show()

from pyspark.sql.functions import to_date

date_issue_df = df.filter(
    to_date(col("check_out_date")) < to_date(col("check_in_date"))
)

date_issue_df.select("check_in_date", "check_out_date").show()

+-------------+--------------+
|check_in_date|check_out_date|
+-------------+--------------+
+-------------+--------------+



In [0]:
# invalid email check

from pyspark.sql.functions import col

invalid_email_df = df.filter(
    (~col("customer_email").like("%@%.%")) |
    (col("customer_email").isNull())
)

invalid_email_df.select("customer_email").show()
invalid_email_df.select("customer_email").count()

+--------------+
|customer_email|
+--------------+
| invalid-email|
| invalid-email|
| invalid-email|
| invalid-email|
| invalid-email|
| invalid-email|
| invalid-email|
| invalid-email|
| invalid-email|
| invalid-email|
| invalid-email|
| invalid-email|
| invalid-email|
| invalid-email|
| invalid-email|
| invalid-email|
| invalid-email|
| invalid-email|
| invalid-email|
| invalid-email|
+--------------+
only showing top 20 rows


401

In [0]:

# Replace invalid email to null
from pyspark.sql.functions import col, when

df_email = df.withColumn(
    "customer_email",
    when(
        (~col("customer_email").like("%@%.%")) |
        (col("customer_email").isNull()),
        None   # invalid → NULL
    ).otherwise(col("customer_email"))
)

df_email.select("customer_email").show()

+--------------------+
|      customer_email|
+--------------------+
|                NULL|
|                NULL|
|                NULL|
|                NULL|
|robert19@example.org|
|tonyaellison@exam...|
|michael60@example...|
|                NULL|
|salastina@example...|
|  swoods@example.com|
|christinearnold@e...|
|                NULL|
|mooreemma@example...|
|                NULL|
|brandi57@example.org|
|                NULL|
|robert35@example.org|
|                NULL|
|  lisa57@example.net|
|  asolis@example.org|
+--------------------+
only showing top 20 rows


In [0]:
# invalid email check

from pyspark.sql.functions import col

invalid_email_df = df_email.filter(
    (~col("customer_email").like("%@%.%")) |
    (col("customer_email").isNull())
)

invalid_email_df.select("customer_email").show()
invalid_email_df.select("customer_email").count()

+--------------+
|customer_email|
+--------------+
|          NULL|
|          NULL|
|          NULL|
|          NULL|
|          NULL|
|          NULL|
|          NULL|
|          NULL|
|          NULL|
|          NULL|
|          NULL|
|          NULL|
|          NULL|
|          NULL|
|          NULL|
|          NULL|
|          NULL|
|          NULL|
|          NULL|
|          NULL|
+--------------+
only showing top 20 rows


401

In [0]:

# Null Value
from pyspark.sql.functions import col, sum

null_counts = df_email.agg(*[sum(col(c).isNull().cast("int")).alias(c) for c in df.columns])
display(null_counts)

booking_id,hotel_id,hotel_city,customer_id,customer_name,customer_email,check_in_date,check_out_date,room_type,num_guests,total_amount,currency,booking_status
2,2,3,1,4,401,267,4,2,3,1,1,3


In [0]:
# Total Amount verify

from pyspark.sql.functions import col

negative_amount_df = df_email.filter(
    col("total_amount").cast("double") < 0
)

negative_amount_df.select("total_amount").show()
negative_amount_df.select("total_amount").count()


+------------+
|total_amount|
+------------+
|     -506.82|
|     -443.72|
|     -158.21|
|     -312.83|
|     -330.26|
|     -430.86|
|     -451.85|
|     -390.17|
|     -367.41|
|     -571.39|
|     -501.18|
|      -339.4|
|     -440.88|
|     -185.68|
|      -436.2|
|      -53.51|
|     -186.78|
|     -543.81|
|     -419.66|
|     -198.66|
+------------+
only showing top 20 rows


176

In [0]:
# Negative Amount Replace to Positive

from pyspark.sql.functions import abs, col

df_amount = df_email.withColumn("total_amount", abs(col("total_amount")))

display(df_amount)   

booking_id,hotel_id,hotel_city,customer_id,customer_name,customer_email,check_in_date,check_out_date,room_type,num_guests,total_amount,currency,booking_status
bc3c2ecb-89b3-4f87-9714-355c5088b35d,1860,Michelleport,1d541501-4a4a-479e-9d6c-91f80cf8452e,Tracy Ortega,null,2026-01-11,2026-01-19,Deluxe,4,252.49,USD,Confirmed
6834f794-1cee-41fe-ba24-d54368d6865e,1058,North Tina,11e3742c-0326-473e-964c-db5ab6de07f2,Karen Coleman,null,2025-10-16,2025-10-20,Suite,2,150.48,INR,Confirmed
e8f146a0-faf2-4c69-9a2b-ce6593d10950,1638,New Frederickville,7abf382a-89b8-4ed5-b5db-fe8c9b66f979,Anthony Alvarez,null,2026-02-06,2026-02-12,Standard,5,197.22,USD,No-Show
83bd8eab-d136-4153-bb30-4d0367b1e106,1980,Port Wesley,917c92b9-9ef1-46fc-9098-db84612c8856,Russell Williams,null,2025-11-11,2025-11-17,Deluxe,2,464.3,EUR,No-Show
c9a791c0-1f34-4c18-b1e6-f1c85ab916c4,1739,North Stevehaven,17485b29-a80e-41a4-8d3a-e783350006f7,Natalie Green,robert19@example.org,2025-02-19,2025-02-27,Standard,2,507.24,EUR,Confirmed
fadd2f9a-ee13-4e4a-98bf-a685669a40d0,1923,Cliffordbury,451192ab-3d5c-480a-b702-ded5aafcbdaa,Jean Chan,tonyaellison@example.com,2025-05-28,2025-06-02,Standard,2,503.82,INR,Confirmeeed
0e62ae16-3c2c-4eb9-ba55-89e6f8e4f3dc,1136,Martintown,707d69f1-a8d5-4665-a092-68fa6b5c64c8,Aaron Young,michael60@example.net,2025-03-11,2025-03-19,Deluxe,1,176.15,USD,Confirmed
3ac87823-3996-49b9-bde7-b9c30ab9cc44,1299,Johnsonland,0567969e-3d96-40f9-9f28-54a917f9c7c6,Yvette Butler,null,2026-05-23,2026-05-26,Suite,4,470.29,INR,Confirmed
c04fd7c4-6cba-4402-8c3f-fa9e82c54b3d,1348,West Jenniferborough,aad9ddf8-6514-41ed-bcae-d4de96982a40,David Hall,salastina@example.com,2026-01-30,2026-02-07,Deluxe,2,506.82,USD,Cancelled
89c4d14c-a632-4d64-95ca-8a1f44f7ed96,1564,South Grant,983b9421-baa4-4ad1-8a63-11b5d03b6a10,Sarah Johnson,swoods@example.com,2026-05-21,2026-05-25,Suite,3,355.99,INR,Confirmed


In [0]:
# Total Amount verify

from pyspark.sql.functions import col

negative_amount_df = df_amount.filter(
    col("total_amount").cast("double") < 0
)

negative_amount_df.select("total_amount").show()
negative_amount_df.select("total_amount").count()


+------------+
|total_amount|
+------------+
+------------+



0

In [0]:
distinct_booking_status_df = df_amount.select("booking_status").distinct()
display(distinct_booking_status_df)

booking_status
Confirmed
No-Show
Confirmeeed
Cancelled
null


In [0]:
from pyspark.sql.functions import col, when

df_booking = df_amount.withColumn(
    "booking_status",
    when(col("booking_status") == "Confirmeeed", "Confirmed")
    .otherwise(col("booking_status"))
)

df_booking.select("booking_status").distinct().show()

+--------------+
|booking_status|
+--------------+
|     Confirmed|
|       No-Show|
|     Cancelled|
|          NULL|
+--------------+



In [0]:
display(df_booking)

booking_id,hotel_id,hotel_city,customer_id,customer_name,customer_email,check_in_date,check_out_date,room_type,num_guests,total_amount,currency,booking_status
bc3c2ecb-89b3-4f87-9714-355c5088b35d,1860,Michelleport,1d541501-4a4a-479e-9d6c-91f80cf8452e,Tracy Ortega,null,2026-01-11,2026-01-19,Deluxe,4,252.49,USD,Confirmed
6834f794-1cee-41fe-ba24-d54368d6865e,1058,North Tina,11e3742c-0326-473e-964c-db5ab6de07f2,Karen Coleman,null,2025-10-16,2025-10-20,Suite,2,150.48,INR,Confirmed
e8f146a0-faf2-4c69-9a2b-ce6593d10950,1638,New Frederickville,7abf382a-89b8-4ed5-b5db-fe8c9b66f979,Anthony Alvarez,null,2026-02-06,2026-02-12,Standard,5,197.22,USD,No-Show
83bd8eab-d136-4153-bb30-4d0367b1e106,1980,Port Wesley,917c92b9-9ef1-46fc-9098-db84612c8856,Russell Williams,null,2025-11-11,2025-11-17,Deluxe,2,464.3,EUR,No-Show
c9a791c0-1f34-4c18-b1e6-f1c85ab916c4,1739,North Stevehaven,17485b29-a80e-41a4-8d3a-e783350006f7,Natalie Green,robert19@example.org,2025-02-19,2025-02-27,Standard,2,507.24,EUR,Confirmed
fadd2f9a-ee13-4e4a-98bf-a685669a40d0,1923,Cliffordbury,451192ab-3d5c-480a-b702-ded5aafcbdaa,Jean Chan,tonyaellison@example.com,2025-05-28,2025-06-02,Standard,2,503.82,INR,Confirmed
0e62ae16-3c2c-4eb9-ba55-89e6f8e4f3dc,1136,Martintown,707d69f1-a8d5-4665-a092-68fa6b5c64c8,Aaron Young,michael60@example.net,2025-03-11,2025-03-19,Deluxe,1,176.15,USD,Confirmed
3ac87823-3996-49b9-bde7-b9c30ab9cc44,1299,Johnsonland,0567969e-3d96-40f9-9f28-54a917f9c7c6,Yvette Butler,null,2026-05-23,2026-05-26,Suite,4,470.29,INR,Confirmed
c04fd7c4-6cba-4402-8c3f-fa9e82c54b3d,1348,West Jenniferborough,aad9ddf8-6514-41ed-bcae-d4de96982a40,David Hall,salastina@example.com,2026-01-30,2026-02-07,Deluxe,2,506.82,USD,Cancelled
89c4d14c-a632-4d64-95ca-8a1f44f7ed96,1564,South Grant,983b9421-baa4-4ad1-8a63-11b5d03b6a10,Sarah Johnson,swoods@example.com,2026-05-21,2026-05-25,Suite,3,355.99,INR,Confirmed


In [0]:
distinct_currency_status_df = df_booking.select("currency").distinct()
display(distinct_currency_status_df)

currency
USD
INR
EUR
null


In [0]:
distinct_room_type_df = df_booking.select("room_type").distinct()
display(distinct_room_type_df)

room_type
Deluxe
Suite
Standard
null


In [0]:
from pyspark.sql.functions import col, trim, initcap

df_trim = df_booking.withColumn(
    "hotel_city",
    initcap(trim(col("hotel_city")))
).withColumn(
    "customer_name",
    initcap(trim(col("customer_name")))
)

display(df_trim)

booking_id,hotel_id,hotel_city,customer_id,customer_name,customer_email,check_in_date,check_out_date,room_type,num_guests,total_amount,currency,booking_status
bc3c2ecb-89b3-4f87-9714-355c5088b35d,1860,Michelleport,1d541501-4a4a-479e-9d6c-91f80cf8452e,Tracy Ortega,null,2026-01-11,2026-01-19,Deluxe,4,252.49,USD,Confirmed
6834f794-1cee-41fe-ba24-d54368d6865e,1058,North Tina,11e3742c-0326-473e-964c-db5ab6de07f2,Karen Coleman,null,2025-10-16,2025-10-20,Suite,2,150.48,INR,Confirmed
e8f146a0-faf2-4c69-9a2b-ce6593d10950,1638,New Frederickville,7abf382a-89b8-4ed5-b5db-fe8c9b66f979,Anthony Alvarez,null,2026-02-06,2026-02-12,Standard,5,197.22,USD,No-Show
83bd8eab-d136-4153-bb30-4d0367b1e106,1980,Port Wesley,917c92b9-9ef1-46fc-9098-db84612c8856,Russell Williams,null,2025-11-11,2025-11-17,Deluxe,2,464.3,EUR,No-Show
c9a791c0-1f34-4c18-b1e6-f1c85ab916c4,1739,North Stevehaven,17485b29-a80e-41a4-8d3a-e783350006f7,Natalie Green,robert19@example.org,2025-02-19,2025-02-27,Standard,2,507.24,EUR,Confirmed
fadd2f9a-ee13-4e4a-98bf-a685669a40d0,1923,Cliffordbury,451192ab-3d5c-480a-b702-ded5aafcbdaa,Jean Chan,tonyaellison@example.com,2025-05-28,2025-06-02,Standard,2,503.82,INR,Confirmed
0e62ae16-3c2c-4eb9-ba55-89e6f8e4f3dc,1136,Martintown,707d69f1-a8d5-4665-a092-68fa6b5c64c8,Aaron Young,michael60@example.net,2025-03-11,2025-03-19,Deluxe,1,176.15,USD,Confirmed
3ac87823-3996-49b9-bde7-b9c30ab9cc44,1299,Johnsonland,0567969e-3d96-40f9-9f28-54a917f9c7c6,Yvette Butler,null,2026-05-23,2026-05-26,Suite,4,470.29,INR,Confirmed
c04fd7c4-6cba-4402-8c3f-fa9e82c54b3d,1348,West Jenniferborough,aad9ddf8-6514-41ed-bcae-d4de96982a40,David Hall,salastina@example.com,2026-01-30,2026-02-07,Deluxe,2,506.82,USD,Cancelled
89c4d14c-a632-4d64-95ca-8a1f44f7ed96,1564,South Grant,983b9421-baa4-4ad1-8a63-11b5d03b6a10,Sarah Johnson,swoods@example.com,2026-05-21,2026-05-25,Suite,3,355.99,INR,Confirmed


In [0]:
df_clean = df_trim.dropna(subset=["check_in_date", "check_out_date"])

In [0]:

# Null Value
from pyspark.sql.functions import col, sum

null_counts = df_clean.agg(*[sum(col(c).isNull().cast("int")).alias(c) for c in df.columns])
display(null_counts)

booking_id,hotel_id,hotel_city,customer_id,customer_name,customer_email,check_in_date,check_out_date,room_type,num_guests,total_amount,currency,booking_status
2,2,3,1,4,401,0,0,2,3,1,1,3


In [0]:
total_count = df_clean.count()
display(total_count)

1729

In [0]:
df_clean.write.format("delta").mode("overwrite").saveAsTable("hotel.silver.hotel_bookings_cleaned")